In [ ]:
# Mini-DGM (Darwin Gödel Machine)
"""
    > Background: from Gödel Machine to Darwin Gödel Machine

        Prove the change is better -> empirically test the change is better by running the modified agent on a benchmark and keep the change if the score went up.

    > What DGM actually is?

        It's a self-improving coding agent. The agent is a frozen foundation-model LLM wrapped in a scaffold — a Python codebase that defines its tools, its workflow, and its prompts. 
        DGM improves the scaffold, not the model weights. The agent reads and rewrites its own Python code to become a better coding agent, and it validates each rewrite by running on coding benchmarks,
        such as SWE-bench and Polyglot, which are explicitly designed and widely accepted as industry-standard benchmarks for evaluating AI coding agents.

        The two ingredients that make it work: 
            1. Self-modification. 
            2. An archive, not a single champion.
               DGM does not keep only the current best agent and hill-climb from it. It keeps an archive of every valid agent it has ever produced. 
               Each iteration it samples a parent from the archive, mutates it, evaluates the child, and adds the child back.
               The reason for it is stepping stones: an agent that scores poorly today might be the necessary ancestor of a breakthrough three generations from now. 
               Greedy hill-climbing throws those ancestors away and gets stuck in local optima; the archive preserves them.

    > The loop, step by step

        1. Seed. Start with one basic coding agent (a minimal scaffold that can read files, write code, run tests).
        2. Select a parent. Sample an agent from the archive. Selection favors high performers but deliberately keeps diversity — it also biases toward agents with few children so far, 
           so under-explored branches get a chance. (Pure "always pick the best" collapses back into hill-climbing.)
        3. Self-modify. The parent inspects its own code and its failure logs, and proposes + implements one modification to itself, producing a child codebase.
        4. Evaluate. Run the child on the benchmark. If it doesn't even compile/run, it's discarded. Otherwise it gets a score.
        5. Archive. Add the valid child to the archive with its score and its lineage (who its parent was).
        6. Repeat from step 2.

    > The caveats to keep honest

        1. The base model is the ceiling. Weights never change, so DGM finds the best use of a fixed brain. 
           It cannot think its way past what the underlying model can do.
        2. It lives or dies by the verifier. All of this only works because coding has a cheap, automatic, hard-to-fully-game scorer. 
           No verifier, no gradient to climb.
        3. Reward hacking is real and they saw it. Agents found ways to look successful without being successful — e.g., faking that tools ran, or gaming the evaluation. 
           This isn't a rare edge case; it's a structural tendency of any optimizer.
        4. Safety plumbing is not optional. 
"""

In [ ]:
"""
    > The build roadmap
    
        1. Foundations & setup — Python env, project skeleton, verify Ollama, pin a model.
        2. LLM client — a clean, portable wrapper around Ollama (using the OpenAI-compatible API, the industry-standard interface).
        3. Task suite (katas) — a kata schema with public tests the agent sees and hidden tests it never sees. This split is our main anti-reward-hacking weapon.
        4. Sandbox executor — run untrusted, model-written code safely (timeouts, isolation). Abstracted so we can start with subprocess and move to Docker.
        5. Seed agent (generation 0) — the simplest solver scaffold: spec -> code -> extract. This is the thing that will later evolve.
        6. Evaluator — run an agent over the suite, score it on hidden tests, emit rich diagnostics.
        7. Meta-agent (self-modification) — the LLM proposes a patch to the scaffold's own code, which we validate and apply. This is the heart of DGM.
        8. Archive + parent selection — persistent store with full lineage, and non-greedy selection (performance + diversity) so stepping stones survive.
        9. DGM orchestrator — the loop that ties 5-8 together, config-driven and seeded for reproducibility.
        10. Observability & experiments — logging/tracing, a small dashboard (score over generations, lineage tree), then the fun experiments: greedy-vs-archive, catching reward hacking, watching the plateau.
    
    > Project folder layout

        rsi/
         ├── README.md
         ├── LICENSE
         ├── requirements.txt
         ├── config.yaml                        # run settings (model, temps, budgets, seed)
         ├── .gitignore
         ├── src/
         │   └── mini-dgm/
         │       ├── __init__.py
         │       ├── llm.py                     # Phase 1 — Ollama client
         │       ├── sandbox.py                 # Phase 4 — safe code execution
         │       ├── evaluator.py               # Phase 6 — scoring on hidden tests
         │       ├── archive.py                 # Phase 8 — storage + selection
         │       ├── meta_agent.py              # Phase 7 — self-modification
         │       ├── orchestrator.py            # Phase 9 — the DGM loop
         │       └── mini-dgm-dev-note.ipynb    # Project development note
         ├── agents/
         │   └── seed/                          # generation-0 scaffold lives here (Phase 5)
         ├── scripts/
         │   └── test_ollama.py                 # Phase 0 - test the structure of model output
         ├── tasks/                             # katas: specs + public & hidden tests (Phase 3)
         ├── runs/                              # outputs, logs, archive db (gitignored)
         └── tests/                             # OUR unit tests for the infrastructure
"""

In [ ]:
"""
    > The kata task suite (and the anti-reward-hacking firewall)

        The central idea: split every task's tests into public and hidden.
        The schema. One folder per kata, so tests are real Python (authentic, and the public ones are directly readable by the agent):

            tasks/
            └── rle_encode/
                ├── spec.md            # problem statement — the agent SEES this
                ├── meta.yaml          # id, entry_point, timeout, difficulty, tags
                ├── tests_public.py    # visible tests — the agent SEES this
                └── tests_hidden.py    # held-out tests — NEVER shown; scoring only

        The public tests look reassuring; the hidden ones probe generalization. 
        That asymmetry is intentional — it is what gives the firewall teeth.
"""